# BQIS v3.5 — Model Audit & Live Simulator
### Tujuan: (1) Verifikasi apakah akurasi XGBoost robust atau cuma menghafal rule generator,
### (2) Bangun simulator "hidup" yang bisa didemoin langsung — nilai tambah sesuai standar pabrik.

**Kenapa notebook ini penting:** Ditemukan bahwa `Historical_Status` di dataset dummy
dihasilkan dari rule IF-ELSE langsung atas parameter yang sama dipakai untuk training.
Ini berarti akurasi 0.98 kemungkinan besar adalah "menghafal aturan", bukan "menemukan pola".
Notebook ini menguji seberapa parah efeknya, dan tetap membangun simulator yang bisa
didemoin dengan jujur soal keterbatasannya.


In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import shap

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report

np.random.seed(42)

## 1. Load Data & Siapkan Fitur
Sama seperti BQIS_01, tapi sekarang kita AUDIT prosesnya, bukan cuma jalanin sekali.

In [20]:
df = pd.read_csv("../data/bqis_biscuit_quality_dataset.csv")

df_model = df.drop(columns=['Sample_ID', 'Batch_Code', 'Test_Date', 'Failure_Category'])
df_model = pd.get_dummies(df_model, columns=['Product_Name'], prefix='Product')
df_model['Historical_Status'] = df_model['Historical_Status'].map({'Pass': 0, 'Fail': 1})

from sklearn.impute import KNNImputer
numeric_cols = ['Moisture_Content_%', 'Fat_Content_%', 'Protein_Content_%', 'Water_Activity_Aw',
                 'Acid_Insoluble_Ash_%', 'Acid_Value_mgKOHg', 'Peroxide_Value',
                 'Lead_Pb_mgkg', 'Cadmium_Cd_mgkg']
imputer = KNNImputer(n_neighbors=5, weights='distance')
df_model[numeric_cols] = imputer.fit_transform(df_model[numeric_cols])

X = df_model.drop(columns=['Historical_Status'])
y = df_model['Historical_Status']
print("Total sampel:", len(X), "| Fail:", y.sum(), f"({y.mean()*100:.1f}%)")

Total sampel: 1000 | Fail: 348 (34.8%)


### buat eda udah ada di v1

## 2. AUDIT #1 — K-Fold Cross-Validation (Bukan 1x Split Saja)
Kalau akurasi konsisten tinggi di SEMUA fold -> modelnya memang stabil (walau kemungkinan
tetap karena rule-based data, bukan berarti buruk, tapi kita perlu tau ini).
Kalau variasinya besar antar fold -> akurasi 0.98 kemarin itu kebetulan dari satu split saja.

In [21]:
model = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                           eval_metric='logloss', random_state=42)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
acc_scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')
f1_scores = cross_val_score(model, X, y, cv=skf, scoring='f1')

print("=== 5-Fold Cross-Validation ===")
print(f"Accuracy per fold : {np.round(acc_scores, 4)}")
print(f"Accuracy mean±std : {acc_scores.mean():.4f} ± {acc_scores.std():.4f}")
print(f"F1-Score per fold : {np.round(f1_scores, 4)}")
print(f"F1-Score mean±std : {f1_scores.mean():.4f} ± {f1_scores.std():.4f}")

if acc_scores.std() < 0.02:
    print("\n>> Std sangat kecil: model SANGAT KONSISTEN di semua fold.")
    print(">> Ini justru memperkuat dugaan rule-based leakage -- kalau model cuma")
    print(">> 'beruntung' di satu split, std antar fold biasanya lebih besar.")

=== 5-Fold Cross-Validation ===
Accuracy per fold : [0.955 0.97  0.97  0.97  0.97 ]
Accuracy mean±std : 0.9670 ± 0.0060
F1-Score per fold : [0.9302 0.9565 0.9565 0.9565 0.9559]
F1-Score mean±std : 0.9511 ± 0.0105

>> Std sangat kecil: model SANGAT KONSISTEN di semua fold.
>> Ini justru memperkuat dugaan rule-based leakage -- kalau model cuma
>> 'beruntung' di satu split, std antar fold biasanya lebih besar.


## 3. AUDIT #2 — Uji Tanpa Fitur Paling "Mencurigakan"
Kalau kita HAPUS fitur yang jadi basis rule generator (moisture, acid_insol_ash, acid_value,
lead, cadmium, tpc, yeast_mold, peroxide), dan model MASIH akurat tinggi, berarti ada sinyal
asli di luar rule sederhana. Kalau akurasinya jatuh drastis, itu konfirmasi: model memang
hanya menghafal rule threshold, bukan menemukan pola kompleks.

##### L note: model menangkap pola antar-parameter

In [22]:
rule_based_features = ['Moisture_Content_%', 'Acid_Insoluble_Ash_%', 'Acid_Value_mgKOHg',
                        'Lead_Pb_mgkg', 'Cadmium_Cd_mgkg', 'Total_Plate_Count_CFUg',
                        'Yeast_Mold_Count_CFUg', 'Peroxide_Value']

X_no_rule = X.drop(columns=rule_based_features)
acc_no_rule = cross_val_score(model, X_no_rule, y, cv=skf, scoring='accuracy')

print("Perbandingan")
print(f"DENGAN fitur rule-based    : accuracy = {acc_scores.mean():.4f}")
print(f"TANPA fitur rule-based     : accuracy = {acc_no_rule.mean():.4f}")
print(f"\nSelisih: {acc_scores.mean() - acc_no_rule.mean():.4f}")
# print("\n>> Kalau selisih BESAR (>0.3): konfirmasi model utamanya belajar rule threshold,")
# print(">> bukan pola implisit/interaksi kompleks antar parameter.")
# print(">> Ini bukan kegagalan -- ini kejujuran metodologis yang WAJIB diungkap ke juri.")

Perbandingan
DENGAN fitur rule-based    : accuracy = 0.9670
TANPA fitur rule-based     : accuracy = 0.7750

Selisih: 0.1920


## 4. Isi SNI_LIMITS dengan Angka yang Ditemukan di Generator
>**PENTING -- baca dulu sebelum pakai:** Angka-angka ini ditemukan sebagai komentar di
>`generate_dummy_data.py` ("Standard SNI compliance limits"), TAPI ini bukan sumber resmi
>terverifikasi -- ini asumsi programmer yang membuat dataset. WAJIB divalidasi ulang ke
>dokumen SNI 2973:2022 resmi sebelum diklaim ke juri sebagai angka baku mutu asli.

Satu angka (`Peroxide_Value` untuk kategori Stability) SECARA EKSPLISIT dikarang
("no SNI limit, but let's say...") -- JANGAN diklaim sebagai angka resmi SNI di manapun.

In [23]:
SNI_LIMITS = {
    'moisture':       {'max': 5.0,  'verified': True, 'source': 'SNI 2973:2022 Tabel 1'},
    'acid_insol_ash': {'max': 0.1,  'verified': True, 'source': 'SNI 2973:2022 Tabel 1'},
    'acid_value':     {'max': 2.0,  'verified': True, 'source': 'SNI 2973:2022 Tabel 1 -- KOREKSI dari generator (sebelumnya 1.0)'},
    'protein':        {'min': 4.5,  'verified': True, 'source': 'SNI 2973:2022 Tabel 1 (varian 4.1/2.7 tergantung jenis produk)'},
    'lead':           {'max': 0.50, 'verified': True, 'source': 'SNI 2973:2022 Tabel 1'},
    'cadmium':        {'max': 0.20, 'verified': True, 'source': 'SNI 2973:2022 Tabel 1'},
    'tin':            {'max': 40,   'verified': True, 'source': 'SNI 2973:2022 Tabel 1 -- BARU, belum ada di dataset'},
    'mercury':        {'max': 0.05, 'verified': True, 'source': 'SNI 2973:2022 Tabel 1 -- BARU, belum ada di dataset'},
    'arsenic':        {'max': 0.50, 'verified': True, 'source': 'SNI 2973:2022 Tabel 1 -- BARU, belum ada di dataset'},
    'tpc':            {'max': None, 'verified': True, 'source': 'SNI 2973:2022 Tabel 2-4 -- BERTINGKAT per jenis produk, perlu mapping Product_Name'},
    'yeast_mold':     {'max': 10000, 'min_acceptable': 500, 'verified': True, 'source': 'SNI 2973:2022 Tabel 4 (khusus biskuit/kukis/wafer/pai)'},
    'peroxide':       {'max': None, 'verified': True, 'source': 'DIKONFIRMASI: bukan parameter SNI 2973:2022 -- tidak ada di Tabel 1'},
}
for k, v in SNI_LIMITS.items():
    if v.get('max') is None and v.get('min') is None and 'peroxide' not in k:
        flag = "ℹstruktur khusus (lihat source)"
    elif k == 'peroxide':
        flag = "❌ BUKAN parameter SNI (dikonfirmasi)"
    elif v.get('verified'):
        flag = "✅ terverifikasi (SNI 2973:2022)"
    else:
        flag = "belum diverifikasi"

    limits = []
    if v.get('min') is not None:
        limits.append(f"min={v['min']}")
    if v.get('min_acceptable') is not None:
        limits.append(f"min_acceptable={v['min_acceptable']}")
    if v.get('max') is not None:
        limits.append(f"max={v['max']}")
    limit = ", ".join(limits) if limits else "no-limit"
    print(f"{k:16s} {limit:<24} [{flag}]")

moisture         max=5.0                  [✅ terverifikasi (SNI 2973:2022)]
acid_insol_ash   max=0.1                  [✅ terverifikasi (SNI 2973:2022)]
acid_value       max=2.0                  [✅ terverifikasi (SNI 2973:2022)]
protein          min=4.5                  [✅ terverifikasi (SNI 2973:2022)]
lead             max=0.5                  [✅ terverifikasi (SNI 2973:2022)]
cadmium          max=0.2                  [✅ terverifikasi (SNI 2973:2022)]
tin              max=40                   [✅ terverifikasi (SNI 2973:2022)]
mercury          max=0.05                 [✅ terverifikasi (SNI 2973:2022)]
arsenic          max=0.5                  [✅ terverifikasi (SNI 2973:2022)]
tpc              no-limit                 [ℹstruktur khusus (lihat source)]
yeast_mold       min_acceptable=500, max=10000 [✅ terverifikasi (SNI 2973:2022)]
peroxide         no-limit                 [❌ BUKAN parameter SNI (dikonfirmasi)]


## 5. Simulasi

In [ ]:
model_final = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                                 eval_metric='logloss', random_state=42)
model_final.fit(X, y)
explainer = shap.TreeExplainer(model_final)

def simulate_prediction(sample_dict, feature_columns=X.columns, model=model_final, explainer=explainer):
    """
    sample_dict: dictionary nilai parameter, contoh:
      {'Moisture_Content_%': 4.2, 'Fat_Content_%': 17.0, ...}
    Kolom yang tidak diisi otomatis diisi median dari data training (bukan 0 -- lebih realistis).
    """
    row = X.median().copy()
    for k, v in sample_dict.items():
        if k in row.index:
            row[k] = v
        else:
            print(f"[WARNING] Kolom '{k}' tidak dikenali, diabaikan.")

    row_df = pd.DataFrame([row])[feature_columns]
    pred = model.predict(row_df)[0]
    proba = model.predict_proba(row_df)[0][1]
    shap_val = explainer.shap_values(row_df)[0]

    label = "FAIL" if pred == 1 else "PASS"
    print(f"{'='*50}")
    print(f"  PREDIKSI: {label}  (confidence Fail: {proba*100:.1f}%)")
    print(f"{'='*50}")

    top_contrib = pd.Series(shap_val, index=feature_columns).sort_values(key=abs, ascending=False).head(5)
    print("\nTop 5 parameter paling berpengaruh ke prediksi ini:")
    for feat, val in top_contrib.items():
        direction = "-> mendorong FAIL" if val > 0 else "-> mendorong PASS"
        print(f"  {feat:28s}: SHAP={val:+.3f}  {direction}")

    return {'prediction': label, 'confidence_fail': round(proba, 4), 'top_factors': top_contrib.to_dict()}

# Contoh pemakaian -- GANTI nilai ini saat demo langsung ke juri:
result = simulate_prediction({
    'Moisture_Content_%': 5.0,      # di atas 5.0 -> berpotensi Fail (Physicochemical)
    'Lead_Pb_mgkg': 0.1,
    'Total_Plate_Count_CFUg': 800,
})

  PREDIKSI: FAIL  (confidence Fail: 62.4%)

Top 5 parameter paling berpengaruh ke prediksi ini:
  Moisture_Content_%          : SHAP=+4.941  -> mendorong FAIL
  Protein_Content_%           : SHAP=-0.805  -> mendorong PASS
  Acid_Insoluble_Ash_%        : SHAP=-0.692  -> mendorong PASS
  Acid_Value_mgKOHg           : SHAP=-0.509  -> mendorong PASS
  Water_Activity_Aw           : SHAP=-0.504  -> mendorong PASS
